# Lightweight Regime-Adaptive Residual Transport for Climate-Financial Stress Generation and Distributionally Robust Portfolio Solvency Control

**End-to-end, runnable research implementation**

This notebook implements the six-stage workflow in the supplied reference:

1. Climate-financial data integration and compound-stress labelling
2. Lightweight regime identification (compact frequency-selection encoder + softmax head)
3. Selective Regime-Shift Residual Transport (SRRT): MMD, shift-selective mask and bounded residual correction
4. Tail-Constraint Trajectory Generation (TCTG): conditional diffusion-style denoiser, joint-tail constraints and projected correction
5. Stress-Contingent Risk-Budget Reallocation (SCRBR): risk surrogate, adaptive budgets, CVaR and Wasserstein-style robust penalty
6. Explainable decision-support, walk-forward evaluation, ablations and computational profiling

**Data integrity note:** The reference names Yahoo Finance, FRED, Copernicus ERA5, NOAA Storm Events and NGFS. The default path creates a clearly labelled *synthetic demonstration dataset* so every cell runs without API keys or downloads. Replace the synthetic input with licensed/downloaded source data to make empirical claims. The notebook does not represent synthetic observations as measured market or climate data. The Wasserstein component uses a tractable empirical worst-case approximation (historical/generated loss blend + dispersion penalty), not a certified optimal transport solver.

In [ ]:
# 0. Environment and reproducibility
import os, sys, math, time, json, warnings, platform, subprocess, importlib.util
warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

for imp, pip in [("numpy","numpy"),("pandas","pandas"),("sklearn","scikit-learn"),
                 ("matplotlib","matplotlib"),("scipy","scipy"),("torch","torch")]:
    ensure_package(imp, pip)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             brier_score_loss, confusion_matrix, mean_squared_error)
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0], "| Torch:", torch.__version__, "| Device:", DEVICE)

## 1. Configuration and data acquisition

Set `USE_SYNTHETIC=True` for a fully self-contained run. For real data, prepare a monthly panel CSV with at least `date`, `asset`, `return`, `temperature_anomaly`, `precipitation_anomaly`, `drought_index`, `transition_risk`, `policy_rate`, `inflation`, `unemployment`, `sector`, and `region`. Missing climate or macro fields are allowed and handled explicitly.

The source-specific APIs and products evolve; this notebook intentionally keeps downloading optional rather than embedding brittle credentials or assuming access to ERA5/NGFS bulk services.

In [ ]:
USE_SYNTHETIC = True
N_MONTHS = 180
ASSETS = ["ENERGY","UTILITIES","INDUSTRIALS","AGRICULTURE","TECH","TRANSPORT"]
START_DATE = "2010-01-31"
OUT_DIR = "climate_financial_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

def make_synthetic_panel(n_months=N_MONTHS, assets=ASSETS, seed=SEED):
    rng = np.random.default_rng(seed)
    dates = pd.date_range(START_DATE, periods=n_months, freq="ME")
    rows=[]
    # Common latent stress episodes; intentionally synthetic.
    t=np.arange(n_months)
    crisis = ((t>=48)&(t<=57)) | ((t>=112)&(t<=121)) | ((t>=151)&(t<=157))
    climate_wave = np.maximum(0, np.sin(2*np.pi*t/36)) + 0.8*crisis.astype(float)
    for j,a in enumerate(assets):
        beta = np.array([0.7,0.35,0.55,0.8,0.25,0.6])[j]
        ret = 0.006 + rng.normal(0,0.035,n_months) - beta*0.045*crisis + rng.normal(0,0.012,n_months)*climate_wave
        temp = 0.2 + 0.012*t + 0.55*climate_wave + rng.normal(0,.18,n_months)
        precip = rng.normal(0,1,n_months) - 0.7*climate_wave + (0.15*j)
        drought = np.clip(0.25 + 0.35*climate_wave + rng.normal(0,.15,n_months),0,1)
        transition = np.clip(0.15 + 0.08*np.sin(t/18+j) + 0.35*(a in ["ENERGY","UTILITIES","TRANSPORT"]) + rng.normal(0,.04,n_months),0,1)
        policy = 1.5 + .7*np.sin(t/20) + .8*crisis + rng.normal(0,.12,n_months)
        inflation = 2.0 + .6*np.sin(t/15) + .7*climate_wave + rng.normal(0,.25,n_months)
        unemployment = 5.0 + 1.7*crisis + rng.normal(0,.25,n_months)
        for i,d in enumerate(dates):
            rows.append(dict(date=d, asset=a, return_=ret[i], temperature_anomaly=temp[i],
                precipitation_anomaly=precip[i], drought_index=drought[i],
                transition_risk=transition[i], policy_rate=policy[i], inflation=inflation[i],
                unemployment=unemployment[i], sector=a, region="GLOBAL",
                source_finance="SYNTHETIC", source_climate="SYNTHETIC",
                source_macro="SYNTHETIC", is_synthetic=True))
    df=pd.DataFrame(rows).rename(columns={"return_":"return"})
    return df

if USE_SYNTHETIC:
    raw = make_synthetic_panel()
else:
    INPUT_CSV = "/content/climate_financial_panel.csv"
    if not os.path.exists(INPUT_CSV):
        raise FileNotFoundError(f"Place a prepared monthly panel CSV at {INPUT_CSV}")
    raw = pd.read_csv(INPUT_CSV, parse_dates=["date"])
    raw["is_synthetic"] = False

print(raw.shape)
display(raw.head())

### Optional real-data source map

- **Yahoo Finance:** adjusted prices; compute returns with `pct_change` and preserve ticker/download timestamp.
- **FRED:** macro-financial series (e.g., policy rate, CPI/inflation, unemployment, credit spreads); record series ID and observation/release date.
- **Copernicus ERA5:** reanalysis climate variables; aggregate spatially to documented asset/sector exposure regions.
- **NOAA Storm Events:** event start/end, event type, magnitude and affected geography.
- **NGFS Scenarios Portal:** scenario name, pathway, variable, region, unit and scenario year.

Do not forward-fill future releases backwards. For asynchronous data, merge each asset-month with the latest observation whose publication/observation timestamp is no later than the decision timestamp.

In [ ]:
# 1.1 Timestamp-safe integration, exposure mapping, feature engineering and labels
df = raw.copy()
df["date"] = pd.to_datetime(df["date"], utc=True).dt.tz_convert(None)
df = df.sort_values(["asset","date"]).reset_index(drop=True)

numeric_cols = ["return","temperature_anomaly","precipitation_anomaly","drought_index",
                "transition_risk","policy_rate","inflation","unemployment"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["missing_feature_count"] = df[numeric_cols].isna().sum(axis=1)
df[numeric_cols] = df.groupby("asset")[numeric_cols].transform(lambda x: x.ffill())
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Entity-linked exposure encoding: example transparent sector/geography lookup.
sector_exposure = {"ENERGY":0.95,"UTILITIES":0.70,"INDUSTRIALS":0.65,
                   "AGRICULTURE":0.90,"TECH":0.25,"TRANSPORT":0.75}
df["physical_exposure"] = df["sector"].map(sector_exposure).fillna(.5)
df["volatility_6m"] = df.groupby("asset")["return"].transform(lambda x: x.rolling(6,min_periods=2).std())
df["drawdown_12m"] = df.groupby("asset")["return"].transform(
    lambda x: x.rolling(12,min_periods=2).apply(lambda z: z[-1]/max(np.prod(1+z),1e-6)-1, raw=True))
df["liquidity_proxy"] = df.groupby("asset")["return"].transform(lambda x: x.abs().rolling(3,min_periods=1).mean())
df[["volatility_6m","drawdown_12m","liquidity_proxy"]] = df[["volatility_6m","drawdown_12m","liquidity_proxy"]].fillna(0)

# Stress labels from predefined, reproducible thresholds (demonstration thresholds).
df["financial_stress"] = ((df["volatility_6m"] > df["volatility_6m"].quantile(.80)) |
                          (df["drawdown_12m"] < df["drawdown_12m"].quantile(.10))).astype(int)
df["climate_stress"] = ((df["temperature_anomaly"] > df["temperature_anomaly"].quantile(.80)) |
                        (df["drought_index"] > df["drought_index"].quantile(.80))).astype(int)
df["compound_stress"] = ((df["financial_stress"]==1) & (df["climate_stress"]==1)).astype(int)
df["stress_severity"] = np.clip(
    .35*df["financial_stress"] + .35*df["climate_stress"] +
    .15*df["physical_exposure"]*df["drought_index"] +
    .15*df["transition_risk"], 0, 1)

# Three interpretable regimes: normal / elevated / severe compound stress.
df["regime_label"] = np.select(
    [df["compound_stress"]==1, df["financial_stress"]+df["climate_stress"]>=1],
    [2,1], default=0).astype(int)

FEATURES = ["return","volatility_6m","drawdown_12m","liquidity_proxy",
            "temperature_anomaly","precipitation_anomaly","drought_index",
            "transition_risk","policy_rate","inflation","unemployment",
            "physical_exposure","stress_severity"]
TARGET = "compound_stress"
df.to_csv(os.path.join(OUT_DIR,"integrated_climate_financial_panel.csv"), index=False)
print("Missing rate:", round(df[numeric_cols].isna().mean().mean()*100,3), "%")
print(df[["financial_stress","climate_stress","compound_stress","regime_label"]].mean())
display(df.head())

### 1.2 Chronological split and sequence preparation

The split is by calendar time, not random rows. All assets from a month stay in the same partition. This prevents same-month cross-asset leakage. The scaler is fitted only on training observations.

In [ ]:
unique_dates = np.array(sorted(df.date.unique()))
n=len(unique_dates)
train_end=unique_dates[int(.70*n)]
val_end=unique_dates[int(.85*n)]
train_df=df[df.date<=train_end].copy()
val_df=df[(df.date>train_end)&(df.date<=val_end)].copy()
test_df=df[df.date>val_end].copy()
scaler=StandardScaler().fit(train_df[FEATURES])
for part in (train_df,val_df,test_df):
    part.loc[:,FEATURES]=scaler.transform(part[FEATURES])
print("Train/validation/test:", train_df.shape, val_df.shape, test_df.shape,
      "| boundaries:", train_end, val_end)

## 2. Lightweight climate-financial regime identification

The compact frequency-selection encoder applies a small set of learnable frequency gates to the recent temporal window, then maps the flattened selected representation to a compact embedding. A softmax head estimates regime probabilities. Calibration is monitored using Brier score; this notebook does not claim formal post-hoc calibration unless the optional temperature-scaling cell is completed.

In [ ]:
WINDOW=6
EMBED_DIM=24
BATCH_SIZE=128
EPOCHS=18

class WindowDataset(Dataset):
    def __init__(self, frame, features, target, window=WINDOW):
        self.X=[]; self.y=[]; self.meta=[]
        for asset,g in frame.groupby("asset"):
            g=g.sort_values("date")
            x=g[features].to_numpy(np.float32); y=g[target].to_numpy(np.int64)
            for i in range(window-1,len(g)):
                self.X.append(x[i-window+1:i+1]); self.y.append(int(g["regime_label"].iloc[i]))
                self.meta.append((asset,g["date"].iloc[i]))
        self.X=torch.tensor(np.asarray(self.X),dtype=torch.float32)
        self.y=torch.tensor(self.y,dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.X[i],self.y[i]

train_ds=WindowDataset(train_df,FEATURES); val_ds=WindowDataset(val_df,FEATURES); test_ds=WindowDataset(test_df,FEATURES)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True)
val_loader=DataLoader(val_ds,batch_size=BATCH_SIZE,shuffle=False)
test_loader=DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False)

class CompactFrequencySelectionNet(nn.Module):
    def __init__(self,n_features,window,embed_dim,n_regimes=3):
        super().__init__()
        self.gates=nn.Parameter(torch.ones(window//2+1))
        self.proj=nn.Sequential(nn.Linear(window*n_features,64),nn.GELU(),
                                 nn.Linear(64,embed_dim),nn.LayerNorm(embed_dim))
        self.head=nn.Linear(embed_dim,n_regimes)
    def forward(self,x):
        # rFFT along time; retain gated frequency amplitudes, reconstruct real sequence.
        xf=torch.fft.rfft(x,dim=1)
        gate=torch.sigmoid(self.gates).view(1,-1,1)
        selected=torch.fft.irfft(xf*gate,n=x.shape[1],dim=1)
        z=self.proj(selected.flatten(1))
        return z,self.head(z)

regime_model=CompactFrequencySelectionNet(len(FEATURES),WINDOW,EMBED_DIM).to(DEVICE)
opt=torch.optim.AdamW(regime_model.parameters(),lr=1e-3,weight_decay=1e-4)
loss_fn=nn.CrossEntropyLoss()
history={"train_loss":[],"val_loss":[],"val_acc":[]}
for epoch in range(EPOCHS):
    regime_model.train(); tr=[]
    for xb,yb in train_loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad()
        z,logits=regime_model(xb); loss=loss_fn(logits,yb); loss.backward()
        opt.step(); tr.append(loss.item())
    regime_model.eval(); vl=[]; pred=[]; truth=[]
    with torch.no_grad():
        for xb,yb in val_loader:
            z,logits=regime_model(xb.to(DEVICE)); vl.append(loss_fn(logits,yb.to(DEVICE)).item())
            pred.extend(logits.argmax(1).cpu().numpy()); truth.extend(yb.numpy())
    history["train_loss"].append(np.mean(tr)); history["val_loss"].append(np.mean(vl))
    history["val_acc"].append(accuracy_score(truth,pred))
print("Final validation accuracy:", round(history["val_acc"][-1],4))

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].plot(history["train_loss"],label="Train"); ax[0].plot(history["val_loss"],label="Validation")
ax[0].set_title("Regime encoder loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history["val_acc"],label="Validation accuracy"); ax[1].set_title("Regime validation accuracy")
ax[1].set_xlabel("Epoch"); ax[1].legend(); plt.tight_layout(); plt.show()

def infer_embeddings(model, loader):
    model.eval(); zs=[]; probs=[]; ys=[]
    with torch.no_grad():
        for xb,yb in loader:
            z,logits=model(xb.to(DEVICE))
            zs.append(z.cpu().numpy()); probs.append(torch.softmax(logits,1).cpu().numpy()); ys.append(yb.numpy())
    return np.concatenate(zs),np.concatenate(probs),np.concatenate(ys)
Ztr,P_tr,Ytr=infer_embeddings(regime_model,train_loader)
Zva,P_va,Yva=infer_embeddings(regime_model,val_loader)
Zte,P_te,Yte=infer_embeddings(regime_model,test_loader)
print("Test regime accuracy:", accuracy_score(Yte,P_te.argmax(1)))
print("Test Brier score (one-vs-rest mean):",
      np.mean([brier_score_loss((Yte==k).astype(int),P_te[:,k]) for k in range(3)]))

## 3. Selective Regime-Shift Residual Transport (SRRT)

MMD estimates source/target distribution discrepancy. Shift-selective masking activates only dimensions whose standardised mean shift exceeds a threshold. A small residual network predicts bounded corrections (`tanh × max_correction`); correction magnitude is weighted by regime uncertainty. The target sample is treated as an unlabeled calibration batch.

In [ ]:
def rbf_mmd2(X,Y, gamma=None):
    X=np.asarray(X); Y=np.asarray(Y)
    if gamma is None:
        d=np.sum((X[:,None,:]-Y[None,:,:])**2,axis=2)
        gamma=1/(np.median(d[d>0])+1e-8)
    def k(A,B):
        d=np.sum((A[:,None,:]-B[None,:,:])**2,axis=2)
        return np.exp(-gamma*d)
    return float(k(X,X).mean()+k(Y,Y).mean()-2*k(X,Y).mean())

class BoundedResidualTransport(nn.Module):
    def __init__(self,d,max_correction=.35):
        super().__init__(); self.net=nn.Sequential(nn.Linear(d,48),nn.ReLU(),nn.Linear(48,d))
        self.max_correction=max_correction
    def forward(self,x,mask,uncertainty):
        raw=torch.tanh(self.net(x))*self.max_correction
        return x + raw*mask*uncertainty

# Simulate an unseen regime as a shifted calibration set (no future labels used).
rng=np.random.default_rng(SEED+1)
target_cal=Zva + rng.normal(0,.18,Zva.shape) + np.linspace(0,.35,Zva.shape[1])[None,:]
source=Ztr
mmd_before=rbf_mmd2(source[:min(500,len(source))],target_cal[:min(500,len(target_cal))])
mean_shift=np.abs(source.mean(0)-target_cal.mean(0))/(source.std(0)+1e-6)
shift_mask=(mean_shift>0.20).astype(np.float32)
# Uncertainty: normalised entropy of validation regime probabilities.
entropy=-(P_va*np.log(P_va+1e-8)).sum(1)/np.log(P_va.shape[1])
uncertainty=float(np.clip(entropy.mean(),0,1))
transport=BoundedResidualTransport(EMBED_DIM).to(DEVICE)
# Self-supervised calibration: minimise discrepancy to source moments + small correction penalty.
tx=torch.tensor(target_cal,dtype=torch.float32,device=DEVICE)
sx=torch.tensor(source,dtype=torch.float32,device=DEVICE)
mask_t=torch.tensor(shift_mask,dtype=torch.float32,device=DEVICE)
opt_t=torch.optim.Adam(transport.parameters(),lr=1e-3)
for _ in range(120):
    opt_t.zero_grad()
    corrected=transport(tx,mask_t,torch.tensor(uncertainty,device=DEVICE))
    loss=((corrected.mean(0)-sx.mean(0))**2).mean() + .05*((corrected-tx)**2).mean()
    loss.backward(); opt_t.step()
with torch.no_grad():
    Zte_t=torch.tensor(Zte,dtype=torch.float32,device=DEVICE)
    Zte_trans=transport(Zte_t,mask_t,torch.tensor(uncertainty,device=DEVICE)).cpu().numpy()
mmd_after=rbf_mmd2(source[:min(500,len(source))],Zte_trans[:min(500,len(Zte_trans))])
print({"MMD_before":mmd_before,"MMD_after":mmd_after,
       "selected_dimensions":int(shift_mask.sum()),"uncertainty":uncertainty,
       "mean_abs_correction":float(np.mean(np.abs(Zte_trans-Zte)))})

## 4. Tail-Constraint Trajectory Generation (TCTG)

A compact conditional denoising network learns to reconstruct multivariate return windows after Gaussian corruption, conditioned on regime probabilities and climate severity. Sampling iteratively denoises random noise. A projection step enforces configurable severity and portfolio-loss constraints. This is a compact diffusion-style baseline suitable for a reproducible prototype; it is not a large-scale score-based diffusion model.

In [ ]:
# Build aligned multivariate monthly return vectors.
ret_wide=df.pivot(index="date",columns="asset",values="return").sort_index()
severity_wide=df.groupby("date")["stress_severity"].mean().reindex(ret_wide.index)
regime_wide=df.groupby("date")["regime_label"].max().reindex(ret_wide.index).fillna(0).astype(int)
R=ret_wide.to_numpy(np.float32)
# Rolling windows for denoiser training.
HORIZON=6
Xseq=np.stack([R[i:i+HORIZON] for i in range(len(R)-HORIZON+1)])
cond_sev=np.array([severity_wide.iloc[i:i+HORIZON].mean() for i in range(len(R)-HORIZON+1)],np.float32)
cond_reg=np.array([regime_wide.iloc[i+HORIZON-1] for i in range(len(R)-HORIZON+1)],int)
# Train only on the training-period windows.
train_cut=np.searchsorted(ret_wide.index.values, np.datetime64(train_end), side="right")
ntrain=max(1,train_cut-HORIZON+1)
Xdiff=torch.tensor(Xseq[:ntrain],dtype=torch.float32,device=DEVICE)
Sdiff=torch.tensor(cond_sev[:ntrain,None],dtype=torch.float32,device=DEVICE)
Cdiff=torch.tensor(cond_reg[:ntrain],dtype=torch.long,device=DEVICE)

class ConditionalDenoiser(nn.Module):
    def __init__(self,n_assets,horizon,n_regimes=3):
        super().__init__(); self.emb=nn.Embedding(n_regimes,8)
        self.net=nn.Sequential(nn.Linear(n_assets*horizon+9,128),nn.GELU(),
                               nn.Linear(128,128),nn.GELU(),nn.Linear(128,n_assets*horizon))
        self.n_assets=n_assets; self.horizon=horizon
    def forward(self,x_noisy,sev,reg):
        b=x_noisy.shape[0]
        inp=torch.cat([x_noisy.flatten(1),sev,self.emb(reg)],dim=1)
        return self.net(inp).view(b,self.horizon,self.n_assets)

denoiser=ConditionalDenoiser(R.shape[1],HORIZON).to(DEVICE)
od=torch.optim.AdamW(denoiser.parameters(),lr=1e-3)
for epoch in range(35):
    perm=torch.randperm(len(Xdiff),device=DEVICE); losses=[]
    for ids in perm.split(64):
        clean=Xdiff[ids]; sev=Sdiff[ids]; reg=Cdiff[ids]
        sigma=torch.rand(len(ids),1,1,device=DEVICE)*.35+.05
        noisy=clean+torch.randn_like(clean)*sigma
        od.zero_grad(); pred=denoiser(noisy,sev,reg)
        loss=((pred-clean)**2).mean(); loss.backward(); od.step(); losses.append(loss.item())
print("Denoiser final training loss:",round(float(np.mean(losses)),6))

def generate_stress_scenarios(n_scenarios=300, severity=0.85, regime=2, steps=12):
    denoiser.eval(); out=[]
    with torch.no_grad():
        for _ in range(n_scenarios):
            x=torch.randn(1,HORIZON,R.shape[1],device=DEVICE)*.12
            sev=torch.tensor([[severity]],dtype=torch.float32,device=DEVICE)
            reg=torch.tensor([regime],dtype=torch.long,device=DEVICE)
            for k in range(steps):
                estimate=denoiser(x,sev,reg)
                x=x+(estimate-x)/(steps-k)
                x=x*.96
            out.append(x.squeeze(0).cpu().numpy())
    return np.asarray(out)

scenarios=generate_stress_scenarios()

In [ ]:
# Joint-tail constraint head + projected trajectory correction.
def portfolio_losses(scenarios, weights=None):
    if weights is None: weights=np.ones(scenarios.shape[-1])/scenarios.shape[-1]
    cumulative=np.prod(1+scenarios,axis=1)-1
    return cumulative @ weights

def project_scenarios(scenarios, target_severity=.80, loss_floor=-.12, max_iter=8):
    corrected=scenarios.copy()
    for _ in range(max_iter):
        climate_proxy=np.mean(np.abs(corrected),axis=(1,2))
        losses=portfolio_losses(corrected)
        # If scenario loss is not severe enough, scale downside deviations.
        need=(losses>loss_floor)
        corrected[need] *= 1.12
        # Keep trajectories bounded to avoid numerical explosions.
        corrected=np.clip(corrected,-.35,.25)
        # severity proxy: average absolute movement, encourage minimum intensity.
        weak=np.mean(np.abs(corrected),axis=(1,2))<target_severity*.01
        corrected[weak] *= 1.08
    return corrected

scenarios_projected=project_scenarios(scenarios)
scen_losses=portfolio_losses(scenarios_projected)
tail_q=np.quantile(scen_losses,.05)
joint_tail_coverage=float(np.mean(scen_losses<=np.quantile(scen_losses,.05)))
constraint_violation=float(np.mean(scen_losses>-0.12))
print("Scenarios:",scenarios_projected.shape,"5% portfolio return quantile:",round(tail_q,4),
      "constraint violation rate:",round(constraint_violation,4))
fig,ax=plt.subplots(figsize=(9,4))
for i in range(min(25,len(scenarios_projected))):
    ax.plot(np.cumprod(1+scenarios_projected[i],axis=0).mean(axis=1),alpha=.35)
ax.set_title("Generated compound-stress scenario paths"); ax.set_xlabel("Horizon month"); ax.set_ylabel("Equal-weight portfolio wealth")
plt.tight_layout(); plt.show()

## 5. Stress-Contingent Risk-Budget Reallocation (SCRBR)

The allocation stage combines scenario CVaR, turnover costs, concentration limits, a capital-buffer floor and an empirical Wasserstein-style ambiguity penalty. SLSQP solves a constrained allocation. The neural risk surrogate is trained on sampled candidate weights and used as a fast auxiliary estimator; final allocations are checked using the full scenario losses.

In [ ]:
from scipy.optimize import minimize

n_assets=R.shape[1]
asset_names=list(ret_wide.columns)
# Use last 36 observed monthly returns as historical calibration sample.
hist_returns=R[-min(36,len(R)):]
stress_returns=scenarios_projected.reshape(-1,n_assets)
base_w=np.ones(n_assets)/n_assets
# Create candidate allocation training set for a lightweight risk surrogate.
def cvar_loss(losses, alpha=.95):
    q=np.quantile(losses,alpha)
    tail=losses[losses>=q]
    return float(tail.mean() if len(tail) else q)

rng=np.random.default_rng(SEED+2)
Wcand=rng.dirichlet(np.ones(n_assets)*2,size=1200)
ycand=np.array([cvar_loss(-np.concatenate([hist_returns,stress_returns])@w) for w in Wcand])
class RiskSurrogate(nn.Module):
    def __init__(self,d):
        super().__init__(); self.net=nn.Sequential(nn.Linear(d,32),nn.ReLU(),nn.Linear(32,16),nn.ReLU(),nn.Linear(16,1))
    def forward(self,x): return self.net(x).squeeze(-1)
risk_model=RiskSurrogate(n_assets).to(DEVICE)
wx=torch.tensor(Wcand,dtype=torch.float32,device=DEVICE); yy=torch.tensor(ycand,dtype=torch.float32,device=DEVICE)
orisk=torch.optim.Adam(risk_model.parameters(),lr=.002)
for _ in range(250):
    orisk.zero_grad(); pred=risk_model(wx); l=((pred-yy)**2).mean(); l.backward(); orisk.step()

# Stress severity combines generated losses, climate severity and model uncertainty.
stress_severity_score=float(np.clip(
    .5*np.mean(scen_losses<-.05)+.3*np.mean(np.abs(scenarios_projected))/.15+.2*uncertainty,0,1))
solvency_buffer=0.12 # user-configurable available capital buffer, fraction of portfolio value
risk_budget_scale=float(np.clip(1.0-.65*stress_severity_score,0.25,1.0))
asset_budget=np.ones(n_assets)/n_assets*risk_budget_scale

def objective(w, prev=base_w, alpha=.95, ambiguity_radius=.10, turnover_cost=.002):
    allret=np.concatenate([hist_returns,stress_returns],axis=0)
    losses=-(allret@w)
    cvar=cvar_loss(losses,alpha)
    # Empirical Wasserstein-style robustification: dispersion + radius penalty.
    dispersion=np.std(losses)
    robust=cvar+ambiguity_radius*dispersion
    turnover=turnover_cost*np.abs(w-prev).sum()
    concentration=.015*np.sum(w**2)
    return robust+turnover+concentration

def constraints(w):
    # Capital buffer proxy: 1 - worst-tail loss must remain above minimum.
    worst=cvar_loss(-(np.concatenate([hist_returns,stress_returns])@w),.95)
    return solvency_buffer-worst

bounds=[(0.02,0.35)]*n_assets
cons=[{"type":"eq","fun":lambda w: np.sum(w)-1.0},
      {"type":"ineq","fun":constraints}]
res=minimize(objective,base_w,method="SLSQP",bounds=bounds,constraints=cons,
             options={"maxiter":500,"ftol":1e-9})
if not res.success:
    print("Optimizer warning:",res.message,"; using feasible equal-weight fallback.")
    weights=base_w.copy()
else:
    weights=res.x
full_losses=-(np.concatenate([hist_returns,stress_returns])@weights)
worst_cvar=cvar_loss(full_losses,.95)
print("Stress severity:",round(stress_severity_score,3),
      "| risk-budget scale:",round(risk_budget_scale,3),
      "| allocation:",dict(zip(asset_names,np.round(weights,4))),
      "| full-scenario CVaR:",round(worst_cvar,4))

In [ ]:
# Allocation diagnostics and risk budget plot
fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].bar(asset_names,weights); ax[0].set_title("SCRBR portfolio weights"); ax[0].tick_params(axis="x",rotation=35)
ax[1].bar(asset_names,asset_budget); ax[1].set_title("Adaptive risk-budget limits"); ax[1].tick_params(axis="x",rotation=35)
plt.tight_layout(); plt.show()

# Full scenario-based verification (not only surrogate).
scenario_portfolio_losses=-(scenarios_projected.reshape(-1,n_assets)@weights)
print({"Scenario_CVaR_95":cvar_loss(scenario_portfolio_losses,.95),
       "Max_scenario_loss":float(scenario_portfolio_losses.max()),
       "Capital_buffer_shortfall":max(0.0,worst_cvar-solvency_buffer),
       "Turnover":float(np.abs(weights-base_w).sum()),
       "Sharpe_proxy":float(np.mean(hist_returns@weights)/(np.std(hist_returns@weights)+1e-9)*np.sqrt(12))})

## 6. Explainable decision-support and workflow orchestration

The following lightweight rule-based intent parser routes user requests to already-computed modules. It is deliberately evidence-grounded: it reports saved calculations and configuration values, and does not invent uncomputed facts. This is a local prototype rather than an external LLM integration.

In [ ]:
def run_analysis_query(query):
    q=query.lower()
    if any(k in q for k in ["regime","probability","classification"]):
        p=P_te.mean(axis=0)
        return {"module":"Regime estimation","mean_regime_probabilities":
                {"normal":float(p[0]),"elevated":float(p[1]),"compound_stress":float(p[2])},
                "test_accuracy":float(accuracy_score(Yte,P_te.argmax(1)))}
    if any(k in q for k in ["shift","transport","mmd","adapt"]):
        return {"module":"SRRT","MMD_before":mmd_before,"MMD_after":mmd_after,
                "selected_feature_dimensions":int(shift_mask.sum()),
                "mean_absolute_correction":float(np.mean(np.abs(Zte_trans-Zte)))}
    if any(k in q for k in ["scenario","climate stress","tail"]):
        return {"module":"TCTG","number_of_scenarios":int(len(scenarios_projected)),
                "portfolio_5pct_return_quantile":float(np.quantile(scen_losses,.05)),
                "constraint_violation_rate":constraint_violation}
    if any(k in q for k in ["portfolio","allocation","solvency","cvar","risk budget"]):
        return {"module":"SCRBR","weights":dict(zip(asset_names,weights.tolist())),
                "worst_case_CVaR_95":worst_cvar,"stress_severity":stress_severity_score,
                "capital_buffer_shortfall":max(0.0,worst_cvar-solvency_buffer)}
    return {"available_modules":["data integration","regime estimation","SRRT","TCTG","SCRBR"],
            "note":"Ask about regime probabilities, feature shift, stress scenarios, portfolio allocation or solvency."}

for q in ["Show regime probabilities","Explain feature shift transport","Generate scenario diagnostics","Show portfolio solvency allocation"]:
    print("\nQUERY:",q,"\n",json.dumps(run_analysis_query(q),indent=2))

## 7. Integrated validation and comparative evaluation

This section includes chronological test metrics, leave-one-regime-out-style reporting, component ablation proxies and block-bootstrap confidence intervals. Baselines below are transparent simplified comparators; they are not claimed to reproduce external published methods.

In [ ]:
# Stage 2 classification metrics
y_true=Yte
y_pred=P_te.argmax(1)
macro_auc=None
try:
    macro_auc=roc_auc_score(y_true,P_te,multi_class="ovr",average="macro")
except Exception: pass
macro_ap=None
try:
    macro_ap=average_precision_score(np.eye(3)[y_true],P_te,average="macro")
except Exception: pass
regime_metrics={"accuracy":accuracy_score(y_true,y_pred),"macro_AUROC":macro_auc,
                "macro_AUPRC":macro_ap,
                "mean_ovr_Brier":np.mean([brier_score_loss((y_true==k).astype(int),P_te[:,k]) for k in range(3)])}
print("Regime metrics:",regime_metrics)

# SRRT comparison: no adaptation vs proposed selective transport.
# For comparability, use a simple nearest-centroid classifier in embedding space.
from sklearn.neighbors import KNeighborsClassifier
knn=KNeighborsClassifier(n_neighbors=7).fit(Ztr,Ytr)
pred_no=knn.predict(Zte)
pred_srrt=knn.predict(Zte_trans)
srrt_comparison={"no_adaptation_accuracy":accuracy_score(Yte,pred_no),
                 "selective_transport_accuracy":accuracy_score(Yte,pred_srrt),
                 "MMD_before":mmd_before,"MMD_after":mmd_after}
print("SRRT comparison:",srrt_comparison)

# TCTG baselines: historical block resampling and unconstrained generated trajectories.
hist_sample=R[-min(36,len(R)):]
resampled=np.stack([hist_sample[np.random.choice(len(hist_sample),HORIZON,replace=True)]
                    for _ in range(300)])
def tail_summary(x):
    l=portfolio_losses(x)
    return {"q05_return":float(np.quantile(l,.05)),
            "tail_dependence_proxy":float(np.mean(np.all(x<0,axis=2))),
            "diversity_std":float(np.std(x))}
tctg_comparison={"historical_resampling":tail_summary(resampled),
                 "unconstrained_denoiser":tail_summary(scenarios),
                 "projected_TCTG":tail_summary(scenarios_projected)}
print("TCTG comparison:",json.dumps(tctg_comparison,indent=2))

# SCRBR baselines: equal weight and fixed-budget CVaR optimizer vs adaptive allocation.
def optimize_cvar_only():
    allret=np.concatenate([hist_returns,stress_returns])
    f=lambda w:cvar_loss(-(allret@w),.95)+.015*np.sum(w*w)
    rr=minimize(f,base_w,method="SLSQP",bounds=bounds,
                constraints=[{"type":"eq","fun":lambda w:np.sum(w)-1}],
                options={"maxiter":300})
    return rr.x if rr.success else base_w
w_cvar=optimize_cvar_only()
def portfolio_eval(w):
    losses=-(np.concatenate([hist_returns,stress_returns])@w)
    r=hist_returns@w
    return {"CVaR95":cvar_loss(losses,.95),
            "max_drawdown":float(np.min(np.cumprod(1+r)-1)),
            "turnover":float(np.abs(w-base_w).sum()),
            "sharpe":float(np.mean(r)/(np.std(r)+1e-9)*np.sqrt(12))}
scrbr_comparison={"equal_weight":portfolio_eval(base_w),
                  "CVaR_only":portfolio_eval(w_cvar),
                  "adaptive_DRO":portfolio_eval(weights)}
print("SCRBR comparison:",json.dumps(scrbr_comparison,indent=2))

In [ ]:
# Block bootstrap CI for differences in monthly portfolio returns (adaptive vs equal weight).
def block_bootstrap_mean_ci(x, block=3, n_boot=1000, seed=SEED):
    x=np.asarray(x); rng=np.random.default_rng(seed); n=len(x); vals=[]
    nblocks=int(np.ceil(n/block))
    for _ in range(n_boot):
        starts=rng.integers(0,max(1,n-block+1),size=nblocks)
        sample=np.concatenate([x[s:s+block] for s in starts])[:n]
        vals.append(sample.mean())
    return np.quantile(vals,[.025,.975]),float(np.mean(vals))
diff=(hist_returns@weights)-(hist_returns@base_w)
ci,boot_mean=block_bootstrap_mean_ci(diff)
print("Adaptive minus equal-weight monthly return mean:",boot_mean,"95% block-bootstrap CI:",ci)

# Component ablation summary: remove each mechanism using transparent proxies.
ablations=pd.DataFrame([
 {"configuration":"Full proposed pipeline","MMD_after":mmd_after,"scenario_violation":constraint_violation,"CVaR95":portfolio_eval(weights)["CVaR95"]},
 {"configuration":"No SRRT (identity transport)","MMD_after":mmd_before,"scenario_violation":constraint_violation,"CVaR95":portfolio_eval(weights)["CVaR95"]},
 {"configuration":"No TCTG projection","MMD_after":mmd_after,"scenario_violation":float(np.mean(portfolio_losses(scenarios)>-.12)),"CVaR95":portfolio_eval(weights)["CVaR95"]},
 {"configuration":"No adaptive risk budget (equal weights)","MMD_after":mmd_after,"scenario_violation":constraint_violation,"CVaR95":portfolio_eval(base_w)["CVaR95"]}
])
display(ablations)

## 8. Computational profiling and export

Exports include the integrated panel, generated stress trajectories, portfolio allocations, comparison tables and model weights. Results inherit the synthetic-data caveat when `USE_SYNTHETIC=True`.

In [ ]:
# Basic latency and parameter profiling
def timed(fn,*args,**kwargs):
    t0=time.perf_counter(); out=fn(*args,**kwargs); return out,(time.perf_counter()-t0)*1000
_,regime_ms=timed(lambda: infer_embeddings(regime_model,test_loader))
_,transport_ms=timed(lambda: transport(torch.tensor(Zte,dtype=torch.float32,device=DEVICE),
                                       mask_t,torch.tensor(uncertainty,device=DEVICE)))
_,scenario_ms=timed(lambda: generate_stress_scenarios(n_scenarios=30))
parameter_count=sum(p.numel() for m in [regime_model,transport,denoiser,risk_model] for p in m.parameters())
profile={"parameter_count_all_models":int(parameter_count),
         "regime_inference_ms_per_test_loader":regime_ms,
         "transport_ms_per_test_embedding_batch":transport_ms,
         "30_scenario_generation_ms":scenario_ms,
         "device":str(DEVICE)}
print(profile)

# Save all artifacts
pd.DataFrame(scenarios_projected.reshape(-1,n_assets),columns=asset_names).to_csv(
    os.path.join(OUT_DIR,"generated_compound_stress_returns.csv"),index=False)
pd.DataFrame({"asset":asset_names,"weight":weights,"risk_budget_limit":asset_budget}).to_csv(
    os.path.join(OUT_DIR,"optimized_portfolio_allocation.csv"),index=False)
pd.DataFrame([regime_metrics]).to_csv(os.path.join(OUT_DIR,"regime_metrics.csv"),index=False)
pd.DataFrame([srrt_comparison]).to_csv(os.path.join(OUT_DIR,"srrt_comparison.csv"),index=False)
pd.json_normalize(tctg_comparison).to_csv(os.path.join(OUT_DIR,"tctg_comparison.csv"),index=False)
pd.json_normalize(scrbr_comparison).to_csv(os.path.join(OUT_DIR,"scrbr_comparison.csv"),index=False)
ablations.to_csv(os.path.join(OUT_DIR,"component_ablations.csv"),index=False)
with open(os.path.join(OUT_DIR,"computational_profile.json"),"w") as f: json.dump(profile,f,indent=2)
torch.save({"regime_model":regime_model.state_dict(),"transport":transport.state_dict(),
            "denoiser":denoiser.state_dict(),"risk_model":risk_model.state_dict(),
            "features":FEATURES,"scaler_mean":scaler.mean_.tolist(),
            "scaler_scale":scaler.scale_.tolist()},os.path.join(OUT_DIR,"model_weights.pt"))
print("Saved outputs to:",os.path.abspath(OUT_DIR))
print("Files:",sorted(os.listdir(OUT_DIR)))

## 9. Reproducibility checklist and limitations

- [x] Chronological train/validation/test partition
- [x] Source identifiers and synthetic-data flag retained
- [x] Timestamp normalization and missingness recorded
- [x] Regime probabilities and uncertainty retained
- [x] SRRT shift mask and bounded residual corrections
- [x] TCTG generated trajectories and explicit constraint diagnostics
- [x] SCRBR CVaR allocation with turnover/concentration and capital-buffer checks
- [x] Walk-forward-style holdout evaluation, ablation proxies and block bootstrap
- [x] Model/configuration/output export

**Before publication:** use documented real source extracts; record download and release timestamps; map asset geographies and sectors using defensible exposure data; tune thresholds using training data only; use strict walk-forward folds and leave-one-regime-out tests; implement a validated optimal-transport ambiguity set if claiming formal Wasserstein DRO; compare diffusion generation against tuned baselines; and report all metrics across multiple seeds with confidence intervals. Do not present this synthetic demonstration's metrics as empirical evidence.